<a href="https://colab.research.google.com/github/fatidevt/data-skills-journey/blob/main/pandas/exercices/grp_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas Tutorial Exercices For Part 9 :
Cleaning Data - Casting Datatypes and Handling Missing Values

# Resume

`pd.to_numeric(df['col'], errors='coerce') `— safer than `astype()` on messy data because errors='coerce' turns anything that can't convert into NaN instead of crashing.

`dropna()` — use when the row is useless without that value (e.g. missing a patient ID — you can't identify the record at all)

`fillna()` — use when you can reasonably substitute a value (e.g. missing salary → fill with column mean; missing city → fill with 'Unknown')

***Rule of thumb:*** if less than 5% of rows have missing values in a column → drop. If more → fill, or investigate why.

`df.isnull().sum() `       # missing count per column

`df.isnull().sum().sum()`  # total missing in entire DataFrame

`df.isnull().mean() * 100` # percentage missing per column — most useful for clients

**Beginner Exercices**

In [ ]:
import pandas as pd
import numpy as np

data = {
    'name': ['Yasmine', 'Omar', None, 'Khalid', 'Imane', 'Hamza', 'Lina'],
    'age': [23, 35, 29, None, 22, 38, 27],
    'city': ['Rabat', 'Casablanca', 'Rabat', 'Fes', None, 'Casablanca', 'Rabat'],
    'salary': ['6000', '8200', '6100', '9500', '3800', None, '5300'],
    'experience': [1.0, 8.0, 4.0, 15.0, 0.5, 10.0, None]
}
df = pd.DataFrame(data)
df

Detect missing values — show the count per column and the total missing in the entire DataFrame.


In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().sum().sum()

Cast the salary column from string to float (it came in as text from a CSV export).


In [ ]:
df.dtypes

In [ ]:
df['salary'].dtype

In [ ]:
df['salary'] = df['salary'].astype(float)
df['salary'].dtype

Drop rows where name is missing — those records are unidentifiable.


In [ ]:
df = df.dropna(subset=['name'])

Fill missing city values with 'Unknown'.


In [ ]:
df['city'] = df['city'].fillna('Unknown')
df

Fill missing experience with the column mean (a reasonable estimate).



In [ ]:
kiki = df['experience'].mean()
df['experience'] = df['experience'].fillna(kiki)
df

**Intermediate Exercises**

In [ ]:
import pandas as pd
import numpy as np

data = {
    'name': ['Yasmine', 'Omar', None, 'Khalid', 'Imane', 'Hamza', 'Lina', 'Youssef'],
    'age': [23, 35, 29, None, 22, 38, 27, 31],
    'city': ['Rabat', 'Casablanca', 'Rabat', 'Fes', None, 'Casablanca', 'Rabat', None],
    'salary': ['6000', '8200', 'unknown', '9500', '3800', None, '5300', '7100'],
    'department': ['Nursing', 'Admin', 'Pharmacy', 'Nursing', None, 'Admin', None, 'Pharmacy'],
    'experience': [1.0, 8.0, 4.0, 15.0, 0.5, 10.0, None, 6.0]
}
df2 = pd.DataFrame(data)
df2

1. Safe casting with messy data

The salary column has 'unknown' as a string mixed in with numeric strings and a None.

Cast it to float safely — without crashing. Fill any resulting NaN with the column median.

In [ ]:
df2['salary'] = pd.to_numeric(df2['salary'], errors='coerce')
df2

In [ ]:
df2['salary'] = df2['salary'].fillna(df2['salary'].mean())
df2

2. Conditional filling
Fill missing department values based on this rule:


If the employee is in 'Rabat' → fill with 'Nursing'

Otherwise → fill with 'Admin'


(Hint: np.where() — you used it in Part 5)

In [ ]:
# np.where(condition, value_if_true, value_if_false)
# Why is_missing in both?
# Without it, you'd overwrite departments that already have values — so you only touch rows where something is actually missing.
is_missing = df2['department'].isna()
df2['department'] = np.where(
    is_missing & (df2['city'] == 'Rabat'),  # IF department is missing AND city is Rabat
    'Nursing',                               # → fill with 'Nursing'
    np.where(                                # ELSE →
        is_missing,                          #   IF department is still missing (other cities)
        'Admin',                             #   → fill with 'Admin'
        df2['department']                    #   ELSE → keep original value
    )
)
df2

The "client audit" scenario

Before delivering any dataset to a client, a professional always runs a missing values audit. Write a clean audit that shows:


* Count of missing values per column
* Percentage of missing values per column
* Only show columns that actually have missing values (filter out the zeros)


*Output should be a clean DataFrame with two columns: missing_count and missing_pct.*

In [ ]:
missing_count = df2.isnull().sum()
missing_pct = (df2.isnull().sum() / len(df2)) * 100
audit_df = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct
})
audit_df = audit_df[audit_df['missing_count'] > 0]
audit_df = audit_df.sort_values(by='missing_count', ascending=False)
print(audit_df)

# Challenge Problem — "The Healthcare Patient Data Cleanup"
A private clinic sends you a messy patient dataset before their monthly report.

In [ ]:
import pandas as pd
import numpy as np

data = {
    'patient_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'name': ['Ahmed', None, 'Fatima', 'Karim', 'Souad', 'Bilal', None, 'Nour'],
    'age': [45, 32, None, 28, 55, 41, 37, None],
    'city': ['Rabat', 'Fes', 'Casablanca', None, 'Rabat', 'Fes', 'Casablanca', 'Rabat'],
    'diagnosis': ['Diabetes', 'Hypertension', 'Diabetes', 'Asthma', None, 'Asthma', 'Hypertension', None],
    'bill_amount': ['1500', '2200', 'unknown', '1800', '3100', None, '2500', '1900'],
    'visits': [3.0, 5.0, 2.0, None, 4.0, 6.0, None, 3.0]
}
df3 = pd.DataFrame(data)
df3

,patient_id,name,age,city,diagnosis,bill_amount,visits
0,101,Ahmed,45.0,Rabat,Diabetes,1500,3.0
1,102,None,32.0,Fes,Hypertension,2200,5.0
2,103,Fatima,NaN,Casablanca,Diabetes,unknown,2.0
3,104,Karim,28.0,None,Asthma,1800,NaN
4,105,Souad,55.0,Rabat,None,3100,4.0
5,106,Bilal,41.0,Fes,Asthma,None,6.0
6,107,None,37.0,Casablanca,Hypertension,2500,NaN
7,108,Nour,NaN,Rabat,None,1900,3.0


**Client instructions:**
1. Run a missing values audit first — show me what's missing before you touch anything.

2. Drop rows where name is missing — unidentifiable patients.

3. Cast bill_amount to float safely — it has some 'unknown' entries.

4. Fill missing bill_amount with the column median.

5. Fill missing age with the column mean, rounded to 1 decimal.

6. Fill missing city with 'Unknown'.

7. Fill missing diagnosis with 'Under Review'.

8. Fill missing visits with 0 — assume no recorded visits means none happened.

9. Reset index cleanly at the end."

*Map your steps before coding — order matters here (audit before touching anything, drop before filling).*

In [ ]:
#1
# missing_count ===>  count missing values per column
missing_count = df3.isnull().sum()

# missing_pct   ===>  percentage missing per column
missing_pct = (df3.isnull().sum() / len(df3)) * 100

# combine into one clean DataFrame
audit_df = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct
})
# keep only columns that actually have missing values
audit_df = audit_df[audit_df['missing_count'] > 0]
audit_df

,missing_count,missing_pct
name,2,25.0
age,2,25.0
city,1,12.5
diagnosis,2,25.0
bill_amount,1,12.5
visits,2,25.0


In [ ]:
#2
df3.dropna(subset=['name'], inplace=True)

In [ ]:
df3

In [ ]:
#3
df3['bill_amount'] = pd.to_numeric(df3['bill_amount'], errors='coerce')
df3

In [ ]:
#4
df3['bill_amount'] = df3['bill_amount'].fillna(df3['bill_amount'].median())
df3

In [ ]:
#5
df3['age'] = df3['age'].fillna(df3['age'].mean())
df3['age'] = df3['age'].round(1)
df3

In [ ]:
#6
df3['city'] = df3['city'].fillna('Unknown')
df3

In [ ]:
#7
df3['diagnosis'] = df3['diagnosis'].fillna('Under Review')
df3

In [ ]:
#8
df3['visits'] = df3['visits'].fillna(0)
df3

In [ ]:
#9
df3 = df3.reset_index(drop=True)